# BioLoc step-by-step pipeline

This notebook loads one synchronized segment using BioLoc's standard loader, runs every pipeline stage explicitly, and keeps the important intermediate results available for inspection and plotting. Edit only the input cell before running all cells.

In [ ]:
from dataclasses import replace
from pathlib import Path

from bioloc import (
    BirdNETClassifier,
    PerchClassifier,
    PipelineConfig,
    associate_tdoas,
    classify_reconstructed_sources,
    estimate_local_tdoas,
    extract_regions_of_interest,
    extract_tdoa_peaks,
    load_recording,
    localize_sources,
    reconstruct_sources,
)
from bioloc.plotting import (
    plot_association_stage,
    plot_histogram_stage,
    plot_local_tdoa_stage,
    plot_localization_stage,
    plot_reconstruction_stage,
    plot_roi_stage,
)

%matplotlib inline

## 1. Input settings

Use the same `root / recorder ID / optional date / WAV` structure documented in the README.

In [ ]:
RECORDING_ROOT = Path("/Users/alejandro/Documents/PhD/Data/VizinaDailyProcessingAlign")
POSITIONS_CSV = RECORDING_ROOT / "REAL_GPS_Audiomoth.csv"
RECORDER_IDS = ["AUD34", "AUD35", "AUD36", "AUD37", "AUD38"]
RECORDING_DATE = "20260308"
RECORDING_TIME = "053501"
START_SECONDS = 459
DURATION_SECONDS = 5
ENVIRONMENT_MARGIN_M = 100.0
EXAMPLE_PAIR = None          # e.g. (1, 3); None selects the first available pair
CLASSIFIER = "birdnet"       # "birdnet" or "perch"
MIN_CONFIDENCE = 0.1
PERCH_MODEL = "perch_8

## 2. Load and standardize the recording

In [ ]:
recording = load_recording(
    root_directory=RECORDING_ROOT,
    positions_csv=POSITIONS_CSV,
    recorder_ids=RECORDER_IDS,
    recording_date=RECORDING_DATE,
    recording_time=RECORDING_TIME,
    start_seconds=START_SECONDS,
    duration_seconds=DURATION_SECONDS,
    environment_margin_m=ENVIRONMENT_MARGIN_M,
)
config = PipelineConfig()
config = replace(
    config,
    classification=replace(
        config.classification,
        backend=CLASSIFIER,
        min_confidence=MIN_CONFIDENCE,
        perch_model=PERCH_MODEL,
    ),
)
recording

## 3. Region-of-interest extraction

In [ ]:
roi = extract_regions_of_interest(recording, config.roi, config.general)
pair = EXAMPLE_PAIR if EXAMPLE_PAIR is not None else next(iter(roi.masks_by_pair))
print("Available pairs:", list(roi.masks_by_pair))
print("Selected pair:", pair, "mask shape:", roi.for_pair(*pair).shape)
plot_roi_stage(recording, roi, config, pair=pair);

## 4. Local TDOA estimation

In [ ]:
local_tdoas = estimate_local_tdoas(
    recording, roi, config.local_tdoa
)
pair_local = local_tdoas.for_pair(*pair)
print("Local delay observations:", pair_local.delays.size)
plot_local_tdoa_stage(recording, local_tdoas, config, pair=pair);

## 5. Histograms and peak picking

In [ ]:
peaks = extract_tdoa_peaks(local_tdoas, config.peaks)
print("Selected-pair peaks:", peaks.for_pair(*pair).peaks)
plot_histogram_stage(peaks, local_tdoas, config);

## 6. Feature construction and TDOA association

In [ ]:
association = associate_tdoas(
    peaks, local_tdoas, recording, config.association
)
print("Feature vectors:", len(association.features))
print("Associated sources:", len(association.sources))
pair_features = association.features_for_pair(*pair)
plot_association_stage(peaks, local_tdoas, association, config);

## 7. Source localization

In [ ]:
localization = localize_sources(
    association, recording, config.localization
)
print("Local positions (m):
", localization.positions)
print("GPS positions [latitude, longitude]:
", localization.gps_positions)
plot_localization_stage(recording, association, localization, config);

## 8. Source separation and reconstruction

In [ ]:
reconstruction = reconstruct_sources(
    recording,
    peaks,
    local_tdoas,
    association,
    localization,
    config.reconstruction,
)
print("Reconstructed sources:", len(reconstruction.sources))
plot_reconstruction_stage(
    recording, local_tdoas, association, reconstruction, config
);

## 9. Source classification

In [ ]:
if config.classification.backend == "birdnet":
    classifier = BirdNETClassifier(
        min_confidence=config.classification.min_confidence
    )
elif config.classification.backend == "perch":
    classifier = PerchClassifier(
        config.classification.perch_model,
        min_confidence=config.classification.min_confidence,
        hop_duration=config.classification.perch_hop_duration,
        warm_up=config.classification.perch_warm_up,
    )
else:
    raise ValueError("CLASSIFIER must be 'birdnet' or 'perch'")

classification = classify_reconstructed_sources(reconstruction, classifier)
for source in classification.sources:
    print(
        source.source_id,
        source.label,
        source.confidence,
        source.detections,
    )

All intermediate variables (`roi`, `local_tdoas`, `peaks`, `association`, `localization`, `reconstruction`, and `classification`) remain available for further inspection.